# Sentence splitting

Sentence splitting using SpaCy:

In [7]:
import spacy
import pandas as pd

reviews = pd.read_csv("training_set/Reviews.csv", encoding="cp1252")
test = pd.read_csv("test_sets/Sentiment-topic-test.tsv", sep="\t")

nlp = spacy.load("en_core_web_sm")

def split_into_sentences(df, text_col="ReviewContent"):
    sentence_rows = []
    for review_id, text in df[text_col].dropna().items():
        doc = nlp(text)
        for sentence_id, sent in enumerate(doc.sents):
            sentence_text = sent.text.strip()

            if sentence_text:
                sentence_rows.append({
                    "review_id": review_id,
                    "sentence_id": sentence_id,
                    "sentence": sentence_text

                })
    return pd.DataFrame(sentence_rows)


sentences_df = split_into_sentences(reviews, text_col="ReviewContent")
for _, row in sentences_df.head(20).iterrows():
    print(f"Review {row['review_id']} | Sentence {row['sentence_id']}: {row['sentence']}")


Review 0 | Sentence 0: Good.
Review 0 | Sentence 1: It IS a page turner.
Review 0 | Sentence 2: You can read this book in one day, two at the most, and the plot drives the whole book.
Review 0 | Sentence 3: The unreliable narrators (there are two besides the main character) are as unlikable as they are unreliable, and there isn't a nice male in the book.
Review 0 | Sentence 4: Entirely plot driven; the characters are paper thin.
Review 0 | Sentence 5: You can figure out who-dunnit by the middle of the book.
Review 0 | Sentence 6: The ending is weak.
Review 0 | Sentence 7: I can't imagine what all the fuss is about, except that it is quick and there are lots of twists and turns, and you can't trust anyone to tell the truth.
Review 1 | Sentence 0: There are no words for how much I loathed this book.
Review 1 | Sentence 1: This was the first audiobook I ever listened to so not sure if something was lost in translation here, if I just disliked the narrators, or if it is truly the book I ha

# VADER sentiment - 1st system

In [8]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download("vader_lexicon", quiet=True)

vader_model = SentimentIntensityAnalyzer()
test_sentiment_scores = []

def vader_sentiment(sentence):
    score = vader_model.polarity_scores(sentence)["compound"]

    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

for sent_id, sent in test[["sentence id", "text"]].itertuples(index=False):
    test_sentiment_scores.append({
        "sentence_id": sent_id,
        "sentence": sent,
        "score": vader_model.polarity_scores(sent),
        "vader_label": vader_sentiment(sent)
    })

vader_df = pd.DataFrame(test_sentiment_scores)

vader_df.head(20)

,sentence_id,sentence,score,vader_label
0,0,It took eight years for Warner Brothers to rec...,"{'neg': 0.215, 'neu': 0.785, 'pos': 0.0, 'comp...",negative
1,1,All the New York University students love this...,"{'neg': 0.0, 'neu': 0.681, 'pos': 0.319, 'comp...",positive
2,2,This Italian place is really trendy but they h...,"{'neg': 0.107, 'neu': 0.773, 'pos': 0.12, 'com...",positive
3,3,"In conclusion, my review of this book would be...","{'neg': 0.0, 'neu': 0.872, 'pos': 0.128, 'comp...",positive
4,4,The story of this movie is focused on Carl Bra...,"{'neg': 0.0, 'neu': 0.844, 'pos': 0.156, 'comp...",positive
5,5,Chris O'Donnell stated that while filming for ...,"{'neg': 0.0, 'neu': 0.865, 'pos': 0.135, 'comp...",positive
6,6,My husband and I moved to Amsterdam 6 years ag...,"{'neg': 0.0, 'neu': 0.879, 'pos': 0.121, 'comp...",positive
7,7,Dame Maggie Smith performed her role excellent...,"{'neg': 0.0, 'neu': 0.76, 'pos': 0.24, 'compou...",positive
8,8,The new movie by Mr. Kruno was shot in New Yor...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",neutral
9,9,"I always have loved English novels, but I just...","{'neg': 0.0, 'neu': 0.818, 'pos': 0.182, 'comp...",positive


# Text blob: sentiment analysis, system 2

In [9]:
from textblob import TextBlob

def textblob_sentiment(polarity, threshold=0.1):

    if polarity > threshold:
        return "positive"
    elif polarity < -threshold:
        return "negative"
    else:
        return "neutral"

test_textblob_scores = []

for sent_id, sent in test[["sentence id", "text"]].itertuples(index=False):
    blob = TextBlob(sent)
    polarity = blob.sentiment.polarity

    test_textblob_scores.append({
        "sentence_id": sent_id,
        "sentence": sent,
        "textblob_polarity": polarity,
        "textblob_subjectivity": blob.sentiment.subjectivity,
        "textblob_label": textblob_sentiment(polarity)
    })

textblob_df = pd.DataFrame(test_textblob_scores)

textblob_df.head(20)

,sentence_id,sentence,textblob_polarity,textblob_subjectivity,textblob_label
0,0,It took eight years for Warner Brothers to rec...,0.000000,0.000000,neutral
1,1,All the New York University students love this...,0.259091,0.413636,positive
2,2,This Italian place is really trendy but they h...,0.375000,0.600000,positive
3,3,"In conclusion, my review of this book would be...",0.500000,1.000000,positive
4,4,The story of this movie is focused on Carl Bra...,0.090000,0.166667,positive
5,5,Chris O'Donnell stated that while filming for ...,0.000000,0.000000,neutral
6,6,My husband and I moved to Amsterdam 6 years ag...,0.287500,0.700000,positive
7,7,Dame Maggie Smith performed her role excellent...,1.000000,1.000000,positive
8,8,The new movie by Mr. Kruno was shot in New Yor...,0.136364,0.454545,positive
9,9,"I always have loved English novels, but I just...",0.350000,0.400000,positive


In [10]:
comparison_df = vader_df.merge(
    textblob_df,
    on=["sentence_id", "sentence"]
)

comparison_df = comparison_df.merge(
    test[["sentence id", "sentiment"]],
    left_on="sentence_id",
    right_on="sentence id"

)
disagreements = comparison_df[
    comparison_df["vader_label"] != comparison_df["textblob_label"]

]
disagreements.head(20)

,sentence_id,sentence,score,vader_label,textblob_polarity,textblob_subjectivity,textblob_label,sentence id,sentiment
0,0,It took eight years for Warner Brothers to rec...,"{'neg': 0.215, 'neu': 0.785, 'pos': 0.0, 'comp...",negative,0.000000,0.000000,neutral,0,negative
5,5,Chris O'Donnell stated that while filming for ...,"{'neg': 0.0, 'neu': 0.865, 'pos': 0.135, 'comp...",positive,0.000000,0.000000,neutral,5,neutral
8,8,The new movie by Mr. Kruno was shot in New Yor...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",neutral,0.136364,0.454545,positive,8,neutral


In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sentiment_labels = ["negative", "neutral", "positive"]

vader_accuracy = accuracy_score(

    comparison_df["sentiment"],
    comparison_df["vader_label"]

)

print("VADER accuracy:", vader_accuracy)
print(classification_report(
    comparison_df["sentiment"],
    comparison_df["vader_label"],
    labels=sentiment_labels,
    zero_division=0
))

vader_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        comparison_df["sentiment"],
        comparison_df["vader_label"],
        labels=sentiment_labels

    ),

    index=[f"true_{label}" for label in sentiment_labels],
    columns=[f"pred_{label}" for label in sentiment_labels]

)
vader_confusion_matrix

VADER accuracy: 0.6
              precision    recall  f1-score   support

    negative       1.00      0.33      0.50         3
     neutral       1.00      0.33      0.50         3
    positive       0.50      1.00      0.67         4

    accuracy                           0.60        10
   macro avg       0.83      0.56      0.56        10
weighted avg       0.80      0.60      0.57        10



,pred_negative,pred_neutral,pred_positive
true_negative,1,0,2
true_neutral,0,1,2
true_positive,0,0,4


In [14]:
textblob_accuracy = accuracy_score(

    comparison_df["sentiment"],
    comparison_df["textblob_label"]

)

print("TextBlob accuracy:", textblob_accuracy)
print(classification_report(
    comparison_df["sentiment"],
    comparison_df["textblob_label"],
    labels=sentiment_labels,
    zero_division=0
))

textblob_confusion_matrix = pd.DataFrame(

    confusion_matrix(
        comparison_df["sentiment"],
        comparison_df["textblob_label"],
        labels=sentiment_labels

    ),
    index=[f"true_{label}" for label in sentiment_labels],
    columns=[f"pred_{label}" for label in sentiment_labels]

)
textblob_confusion_matrix


TextBlob accuracy: 0.5
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         3
     neutral       0.50      0.33      0.40         3
    positive       0.50      1.00      0.67         4

    accuracy                           0.50        10
   macro avg       0.33      0.44      0.36        10
weighted avg       0.35      0.50      0.39        10



/Users/ilincafarca/Developer/TextMining2026/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ilincafarca/Developer/TextMining2026/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ilincafarca/Developer/TextMining2026/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

,pred_negative,pred_neutral,pred_positive
true_negative,0,1,2
true_neutral,0,1,2
true_positive,0,0,4


# NERC 

Function extracting the entities:

In [ ]:
def extract_entities(sentence):
    doc = nlp(sentence)

    # entities that are more likely to be relevant
    useful_labels = ["PERSON", "ORG", "GPE", "LOC", "PRODUCT", "EVENT", "WORK_OF_ART"]
    entities = []

    # extract entities with their labels
    for ent in doc.ents:
        if ent.label_ in useful_labels:
            entities.append({
                "text": ent.text,
            "label": ent.label_
        })

    return entities

Apply NERC to all sentences:

In [ ]:
# entity extraction to each sentence
for sentence in reviews["ReviewContent"].dropna().head(20):
    entities = extract_entities(sentence)
    print(f"Sentence: {sentence}")
    print(f"Extracted Entities: {entities}\n")
    


Count how many sentences have entities and print only the sencences where entities were found:

In [ ]:
# display sentences with their entities
sentences_df["number_of_entities"] = sentences_df["named_entities"].apply(len)
sentences_df["number_of_entities"].value_counts()

# show sentences that contain at least one entity
sentences_with_entities = sentences_df[sentences_df["number_of_entities"] > 0]
sentences_with_entities[["review_id", "sentence_id", "sentence", "entities_display"]].head(20)

,review_id,sentence_id,sentence,entities_display
23,2,2,This is serviceable writing at best - about th...,Fiction Press (ORG); MFA (ORG)
32,2,11,The book is told from the POV of three differe...,POV (ORG)
35,2,14,"Then we're supposed to care that Rachel, who d...",Rachel (PERSON)
36,2,15,"I didn't care about Rachel, didn't care about ...",Rachel (PERSON); MIA (ORG)
39,2,18,"Oh, right, she's the woman who is responsible ...",Rachel (PERSON); Rachel (PERSON); Tom (PERSON)
44,2,23,I've heard it was compared to Gone Girl but ot...,Gone Girl (PRODUCT)
47,2,26,That kind of 'Bwahahaha I did it!',Bwahahaha (PERSON)
50,2,29,But I guess the biggest thing that bothered me...,Rachel (PERSON)
61,2,40,"It seems if books are bought by book clubs, th...",Shades of Grey).Here's (WORK_OF_ART)
65,3,1,Plot dragged on and ending was very unsubstant...,"Plot (ORG); the ""Gone Girl (PRODUCT)"


In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Topic analysis 

In [29]:
# the possible topics
topic_profiles = {
    "book": "book novel author writer reading pages chapter plot story character literature ending prose",
    "movie": "movie film cinema actor actress director scene screen role cast trailer script",
    "restaurant": "restaurant food menu dish service table dinner lunch meal waiter chef atmosphere"
}

topic_labels = list(topic_profiles.keys())

# TF-IDF vectorizer 
topic_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

# the topic profile matrix
topic_profile_matrix = topic_vectorizer.fit_transform(topic_profiles.values())

Topic prediction function:

In [30]:
# function to predict the topic of a sentence
def predict_topic(sentence):
    sentence_matrix = topic_vectorizer.transform([sentence])
    similarities = cosine_similarity(sentence_matrix, topic_profile_matrix)

    best_topic_index = similarities.argmax()
    predicted_topic = topic_labels[best_topic_index]
    confidence_score = similarities.max()

    return predicted_topic, confidence_score

Topic analysis per sentence:

In [31]:
topic_predictions = sentences_df["sentence"].apply(predict_topic)

# add the predicted topic and confidence score
sentences_df["topic_label"] = topic_predictions.apply(lambda x: x[0])
sentences_df["topic_score"] = topic_predictions.apply(lambda x: x[1])

The first 20 topic results:

In [32]:
sentences_df[[
    "review_id",
    "sentence_id",
    "sentence",
    "topic_label",
    "topic_score"
]].head(20)

,review_id,sentence_id,sentence,topic_label,topic_score
0,0,0,Good.,book,0.000000
1,0,1,It IS a page turner.,book,0.000000
2,0,2,"You can read this book in one day, two at the ...",book,0.268328
3,0,3,The unreliable narrators (there are two beside...,book,0.282843
4,0,4,Entirely plot driven; the characters are paper...,book,0.200000
5,0,5,You can figure out who-dunnit by the middle of...,book,0.200000
6,0,6,The ending is weak.,book,0.200000
7,0,7,"I can't imagine what all the fuss is about, ex...",book,0.000000
8,1,0,There are no words for how much I loathed this...,book,0.200000
9,1,1,This was the first audiobook I ever listened t...,book,0.200000


# Topic analysis on the test set

In [45]:
# load the test set
test_sent_topic = pd.read_csv("test_sets/Sentiment-topic-test.tsv", sep="\t")

# the distribution of topics in the test set
test_sent_topic["topic"].value_counts()
topic_predictions = test_sent_topic["text"].apply(predict_topic)

# add the predicted topic and confidence score to the test set
test_sent_topic["predicted_topic"] = topic_predictions.apply(lambda x: x[0])
test_sent_topic["topic_score"] = topic_predictions.apply(lambda x: x[1])

The results:

In [37]:
test_sent_topic[[
    "sentence id",
    "text",
    "topic",
    "predicted_topic",
    "topic_score"
]]

,sentence id,text,topic,predicted_topic,topic_score
0,0,It took eight years for Warner Brothers to rec...,movie,movie,0.208514
1,1,All the New York University students love this...,restaurant,restaurant,0.208514
2,2,This Italian place is really trendy but they h...,restaurant,restaurant,0.361158
3,3,"In conclusion, my review of this book would be...",book,book,0.200000
4,4,The story of this movie is focused on Carl Bra...,movie,movie,0.147442
5,5,Chris O'Donnell stated that while filming for ...,movie,movie,0.208514
6,6,My husband and I moved to Amsterdam 6 years ag...,restaurant,book,0.000000
7,7,Dame Maggie Smith performed her role excellent...,movie,movie,0.208514
8,8,The new movie by Mr. Kruno was shot in New Yor...,movie,movie,0.147442
9,9,"I always have loved English novels, but I just...",book,book,0.000000


# Evaluate topic accuracy: 

In [46]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

topic_accuracy = accuracy_score(
    test_sent_topic["topic"],
    test_sent_topic["predicted_topic"]
)

print("Topic accuracy:", topic_accuracy)

print(classification_report(
    test_sent_topic["topic"],
    test_sent_topic["predicted_topic"]
))

labels = ["book", "movie", "restaurant"]

topic_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        test_sent_topic["topic"],
        test_sent_topic["predicted_topic"],
        labels=labels
    ),
    index=[f"true_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels]
)

topic_confusion_matrix

Topic accuracy: 0.9
              precision    recall  f1-score   support

        book       0.67      1.00      0.80         2
       movie       1.00      1.00      1.00         5
  restaurant       1.00      0.67      0.80         3

    accuracy                           0.90        10
   macro avg       0.89      0.89      0.87        10
weighted avg       0.93      0.90      0.90        10



,pred_book,pred_movie,pred_restaurant
true_book,2,0,0
true_movie,0,5,0
true_restaurant,1,0,2
